In [27]:
import pandas as pd
import random
import time
random.seed(17)

In [28]:
num_activities = 4
max_tries = 3
prob_blink = 0.1
policy = {'num_a1':2,'num_a2':3,'num_a3':3,'num_a4':3}

In [29]:
activities = [f'a{i+1}' for i in range(num_activities)]
print(activities)

['a1', 'a2', 'a3', 'a4']


In [30]:
activity_types = {}
for a in activities:
    activity_types[a] = random.sample(['flyby','mower','circle'],1)[0]
#activity_types = {'a1': 'circle', 'a2': 'mower', 'a3': 'mower'}
print(activity_types)

{'a1': 'circle', 'a2': 'mower', 'a3': 'mower', 'a4': 'mower'}


In [31]:
num_regions = 2**num_activities
coverage = {}
for i in range(num_regions):
    if i>0:
        coverage[f'r{i}'] = [a for j,a in enumerate(activities) if 2**j & i > 0]
regions = [r for r in coverage.keys()]
region_probs = [0.75+random.random() for r in coverage]
#region_probs = [0.2729508555208528,
#                0.08230497109888947,
#                0.21771037819028088,
#                0.20012333254493608,
#                0.1879500816892201,
#                0.03130555865756731,
#                0.007654822298253374]
#region_probs = [0,0,0,0,0,0,1] # always vis to all
region_probs = [p / sum(region_probs) for p in region_probs]
assert abs(sum(region_probs)-1)<1e-10
pdf_region = pd.DataFrame({'region': coverage.keys(),
                           'prob': region_probs})
pdf_region['coverage'] = [coverage[r] for r in regions]
pdf_region

,region,prob,coverage
0,r1,0.054142,[a1]
1,r2,0.078956,[a2]
2,r3,0.075733,"[a1, a2]"
3,r4,0.073502,[a3]
4,r5,0.044795,"[a1, a3]"
5,r6,0.040461,"[a2, a3]"
6,r7,0.059065,"[a1, a2, a3]"
7,r8,0.077929,[a4]
8,r9,0.052202,"[a1, a4]"
9,r10,0.065181,"[a2, a4]"


In [32]:
pdf_app = pd.DataFrame({'appearance':['blend','neutral','obvious'],
                        'prob' : [0.2,0.4,0.4]})
assert sum(pdf_app['prob'])==1
pdf_app

,appearance,prob
0,blend,0.2
1,neutral,0.4
2,obvious,0.4


In [33]:
pdf_occ = pd.DataFrame({'occlusion':['easy','med','hard'],
                        'prob' : [0.5,0.3,0.2]})
assert sum(pdf_occ['prob'])==1
pdf_occ

,occlusion,prob
0,easy,0.5
1,med,0.3
2,hard,0.2


In [34]:
prob_occ_a_given_occlusion = {'circle':{'hard':0.5,'med':0.3,'easy':0.1},
                              'mower':{'hard':0.7,'med':0.5,'easy':0.3},
                              'flyby':{'hard':0.9,'med':0.7,'easy':0.5},}

In [35]:
#prob_occ_a_given_occlusion = {'circle':{'hard':0.,'med':0.,'easy':0.},
#                              'mower':{'hard':0.,'med':0.,'easy':0.},
#                              'flyby':{'hard':0.,'med':0.,'easy':0.},}

In [36]:
prob_indis_a_given_appearance = {'blend':0.9,'neutral':0.5,'obvious':0.1}

In [37]:
#prob_indis_a_given_appearance = {'blend':0.,'neutral':0.,'obvious':0.}

In [38]:
option_set = [tuple([((2**i)&n) > 0 for i in range(num_activities)]) for n in range(2**num_activities)]

In [39]:
random.seed(time.time())
num_batches = 8
num_trials = 80000 #000
data_store = []
for b in range(num_batches):
    # set up counters
    appearance_dict = dict([(k,0) for k in pdf_app['appearance']])
    occlusion_dict = dict([(k,0) for k in pdf_occ['occlusion']])
    region_dict = dict([(r,0) for r in regions])
    #occ_a_dict = dict([(op,0) for op in option_set])
    #indis_a_dict = dict([(op,0) for op in option_set])
    det_a_dict = dict([(op,0) for op in option_set])
    det_counter = 0
    for t in range(num_trials):
        #print(f'Trial {t}')
        appearance = random.choices(pdf_app['appearance'],weights=pdf_app['prob'],k=1)[0]
        appearance_dict[appearance] += 1
        #print([f'{k}:{appearance_dict[k]/(t+1):.2f}' for k in appearance_dict])
        occlusion = random.choices(pdf_occ['occlusion'],weights=pdf_occ['prob'],k=1)[0]
        occlusion_dict[occlusion] += 1
        #print([f'{k}:{occlusion_dict[k]/(t+1):.2f}' for k in occlusion_dict])
        region = random.choices(pdf_region['region'],weights=pdf_region['prob'],k=1)[0]
        region_dict[region] += 1
        #print([f'{k}:{region_dict[k]/(t+1):.2f}' for k in region_dict])
        det_outcome = dict((a,False) for a in activities)
        for a in activities:
            if policy['num_'+a]>0:
                if a not in coverage[region]:
                    continue # out of range
                occ_a = (random.random() < prob_occ_a_given_occlusion[activity_types[a]][occlusion])
                if occ_a == True:
                    continue # occluded
                indis_a = (random.random() < prob_indis_a_given_appearance[appearance])
                if indis_a == True:
                    continue # indistinguishable
                for look in range(policy['num_'+a]):
                    if random.random() < prob_blink:
                        continue # missed it
                    det_outcome[a] = True
                #break # found it - no need to keep testing
        det_a_dict[tuple([det_outcome[a] for a in activities])] += 1
        if any([det_outcome[a] for a in activities]):
            det_counter+=1
    print(f'Batch {b} overall det prob {det_counter/(t+1):.4f}')
    print('-'*20)
    #print([f'{k}:{appearance_dict[k]/(t+1):.2f}' for k in appearance_dict])
    #print([f'{k}:{occlusion_dict[k]/(t+1):.2f}' for k in occlusion_dict])
    #print([f'{k}:{region_dict[k]/(t+1):.2f}' for k in region_dict])
    #print('-'*20)
    #for k in det_a_dict:
    #    print(f"{k}:{det_a_dict[k]/(t+1):.4f}")
    data_store.append({'det_prob':det_counter/(t+1)})
print(policy)
print([d['det_prob'] for d in data_store])
average_prob_det = sum([d['det_prob'] for d in data_store])/num_batches
print(f'Average {average_prob_det:.4f}')

Batch 0 overall det prob 0.5270
--------------------
Batch 1 overall det prob 0.5273
--------------------
Batch 2 overall det prob 0.5279
--------------------
Batch 3 overall det prob 0.5278
--------------------
Batch 4 overall det prob 0.5305
--------------------
Batch 5 overall det prob 0.5288
--------------------
Batch 6 overall det prob 0.5284
--------------------
Batch 7 overall det prob 0.5283
--------------------
{'num_a1': 2, 'num_a2': 3, 'num_a3': 3, 'num_a4': 3}
[0.52695, 0.52735, 0.5279125, 0.52775, 0.5304875, 0.5288, 0.5283875, 0.5282875]
Average 0.5282
